# Structured Output in LangChain

Structured output is a feature in LangChain that allows a language model to return responses in a predefined format instead of free-form text. Rather than generating paragraphs, the model produces well-organized data that follows a specific schema, such as JSON or a Pydantic model. This makes the output easier to validate, parse, and integrate into applications without requiring additional text processing.

Structured outputs are commonly used when applications need predictable and machine-readable responses. For example, they are useful for extracting information from documents, generating API-ready JSON, creating database records, classifying text, or returning objects with specific fields. By defining the expected structure in advance, developers can reduce errors, improve consistency, and ensure that the model always returns data in the required format.

## When to Use Structured Output

Use structured output when your application needs data in a fixed format rather than plain text. Common use cases include:

- Extracting information from documents
- Returning JSON for APIs
- Creating database records
- Form filling and data validation
- Text classification
- Sentiment analysis
- Generating invoices or reports
- Building AI agents that exchange structured data
- Integrating AI with backend services
- Any workflow that requires consistent, machine-readable responses

## Example

```python
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int

structured_model = model.with_structured_output(Person)

result = structured_model.invoke(
    "John is 28 years old."
)

print(result)
```

**Output**

```python
Person(
    name="John",
    age=28
)
```

With structured output, LangChain automatically validates the model's response against the defined schema, making AI applications more reliable, predictable, and easier to integrate with production systems.

### Pydantic

**Pydantic** is a Python library used to define and validate structured data using Python classes. In LangChain, it is commonly used with structured output to specify the exact format that the language model should return. Pydantic automatically validates data types, ensures required fields are present, and converts valid input into Python objects, making AI applications more reliable and easier to work with.

In [8]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.0,
    max_retries=2,
    # other params...
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000137DD44B750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000137DD4D4190>, model_name='openai/gpt-oss-120b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [9]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title: str= Field(description="The title of the movie")
    year: int= Field(description="The year the movie was released")
    genre: str= Field(description="The genre of the movie")
    director:str=Field(description="The director of the movie")
    rating:float =Field(description="The rating of the movie")
    

In [10]:
model_structured = model.with_structured_output(Movie)
model_structured


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000137DD44B750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000137DD4D4190>, model_name='openai/gpt-oss-120b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'de

In [12]:
response = model_structured.invoke("Provide me details information about Movie called Inception")
response

Movie(title='Inception', year=2010, genre='Science Fiction', director='Christopher Nolan', rating=8.8)

## Meessage output along wih Parsed Structured

In [14]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    """A movie with details and Structured output"""
    title: str= Field(...,description="The title of the movie")
    year: int= Field(...,description="The year the movie was released")
    genre: str= Field(...,description="The genre of the movie")
    director:str=Field(...,description="The director of the movie")
    rating:float =Field(...,description="The rating of the movie")

model_structured = model.with_structured_output(Movie,include_raw=True)
res =model_structured.invoke("Provide me details information about Movie called Inception")
res

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about movie "Inception". We have a function to output structured movie details: director, genre, rating, title, year. We should call the function with appropriate values. Inception: director Christopher Nolan, genre Sci-Fi/Action (maybe "Science Fiction"), rating maybe 8.8 (IMDb). Title "Inception". Year 2010. Provide details. Use function.', 'tool_calls': [{'id': 'fc_57c49129-1020-44a9-b8a9-b99558bd2674', 'function': {'arguments': '{"director":"Christopher Nolan","genre":"Science Fiction","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 176, 'total_tokens': 318, 'completion_time': 0.297881902, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.006968846, 'prompt_tokens_details': None, 'queue_time': 0.286874993, 'total_time': 0.304850748}, 'model_name': 'openai

### Nested Structured

In [15]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name: str= Field(...,description="The name of the actor")
    age: int= Field(...,description="The age of the actor")
    nationality: str= Field(...,description="The nationality of the actor")
class MovieDetails(BaseModel):
    title: str= Field(...,description="The title of the movie")
    year: int= Field(...,description="The year the movie was released")
    genre: str= Field(...,description="The genre of the movie")
    director:str=Field(...,description="The director of the movie")
    rating:float =Field(...,description="The rating of the movie")
    actors: list[Actor] = Field(...,description="The actors in the movie")
    budget:float|None=Field(None,description="The budget of the movie in USD")

structure_data = model.with_structured_output(MovieDetails)
res = structure_data.invoke("Provide me details information about Movie called Inception")
res

MovieDetails(title='Inception', year=2010, genre='Science Fiction, Thriller', director='Christopher Nolan', rating=8.8, actors=[Actor(name='Leonardo DiCaprio', age=36, nationality='American'), Actor(name='Joseph Gordon-Levitt', age=29, nationality='American'), Actor(name='Elliot Page', age=23, nationality='Canadian'), Actor(name='Tom Hardy', age=33, nationality='British'), Actor(name='Ken Watanabe', age=51, nationality='Japanese'), Actor(name='Marion Cotillard', age=35, nationality='French')], budget=160000000.0)

### TypedDict

**TypedDict** is a feature from Python's `typing` module that allows you to define the expected structure of a dictionary with specific keys and value types. Unlike Pydantic, `TypedDict` provides **type checking only** and does **not perform runtime validation**. It is commonly used in LangChain and LangGraph to define the shape of state, configuration, and message data while improving code readability and IDE support.

In [18]:
from typing_extensions import TypedDict,Annotated

class Actor(TypedDict):
    """A dictionary representing an actor."""
    name: Annotated[str,..., "The name of the actor"]
    age: Annotated[int,..., "The age of the actor"]
    nationality: Annotated[str,..., "The nationality of the actor"]

class MovieDict(TypedDict):
    """A dictionary representing movie details."""
    title: Annotated[str,..., "The title of the movie"]
    year: Annotated[int,..., "The year the movie was released"]
    genre: Annotated[str,..., "The genre of the movie"]
    director: Annotated[str,..., "The director of the movie"]
    rating: Annotated[float,..., "The rating of the movie"]
    budget: Annotated[float | None, "The budget of the movie in USD"]
    actors: Annotated[list[Actor], "The actors in the movie"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Please provide the details of the movie 3 Idiots")
response


{'actors': [{'age': 58, 'name': 'Aamir Khan', 'nationality': 'Indian'},
  {'age': 53, 'name': 'R. Madhavan', 'nationality': 'Indian'},
  {'age': 44, 'name': 'Sharman Joshi', 'nationality': 'Indian'},
  {'age': 43, 'name': 'Kareena Kapoor', 'nationality': 'Indian'},
  {'age': 64, 'name': 'Boman Irani', 'nationality': 'Indian'}],
 'budget': 6500000,
 'director': 'Rajkumar Hirani',
 'genre': 'Comedy drama',
 'rating': 8.4,
 'title': '3 Idiots',
 'year': 2009}

In [20]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### Dataclasses

**Dataclasses** are a Python feature that makes it easy to create classes whose main purpose is to store data. By adding the `@dataclass` decorator, Python automatically generates methods such as `__init__()`, `__repr__()`, and `__eq__()`, reducing the amount of code you need to write. Dataclasses are lightweight and are commonly used when you need to organize related data into objects without requiring automatic validation.

#### Example

```python
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int

person = Person(
    name="John",
    age=25
)

print(person)
```

**Output**

```python
Person(name='John', age=25)
```

#### When to Use

- Storing related data in a simple object
- Configuration objects
- Application state
- API response objects
- Data transfer between functions or classes

> **Note:** Unlike Pydantic, dataclasses do **not** validate data types at runtime. They simply store the values you provide.

In [22]:
from langchain.agents import create_agent
from pydantic import BaseModel,Field

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(model, response_format=ContactInfo)

result = agent.invoke({'messages': [{'role': 'user', 'content': 'Please provide your contact information'}]})
result["structured_response"]


ContactInfo(name='ChatGPT', email='chatgpt@example.com', phone='+1-800-555-1234')

In [25]:
from dataclasses import dataclass
from langchain.agents import create_agent
@dataclass
class ContactInfo:
    """Contact Information for a Agent"""
    name: str
    email: str
    phone: str

agent =  create_agent(model,response_format=ContactInfo)
res = agent.invoke({'messages': [{'role': 'user', 'content': 'Please provide your contact information'}]})
res
res["structured_response"]

ContactInfo(name='ChatGPT', email='support@example.com', phone='123-456-7890')